### TDSP · Fase 3: Modelado sin balancear

Primera ejecución: modelos entrenados con la proporción original de clases.

- **Entrada:** `Sample_Data/For_Modeling/train_clean.csv`
- **Salida:** `Code/Modeling/results/cv_unbalanced.csv`
- **Variable objetivo:** `Revenue`

#### Librerías

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, make_scorer, precision_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

#### Semilla

In [2]:
# Semilla para reproducibilidad. Puede modificarse o eliminarse.
SEED = 1
np.random.seed(SEED)

#### Rutas

Relativas a la carpeta del notebook.

In [3]:
FOR_MODELING_DIR = Path("../../Sample_Data/For_Modeling")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

#### Carga de datos

In [4]:
train = pd.read_csv(FOR_MODELING_DIR / "train_clean.csv")

TARGET = "Revenue"
X = train.drop(columns=["row_id", TARGET])
y = train[TARGET]

print(f"Sesiones: {len(X):,} | Compras: {y.sum():,} ({y.mean():.1%})")

Sesiones: 9,864 | Compras: 1,526 (15.5%)


#### Conjuntos de variables

Cada modelo se entrena con todas las variables y sin `PageValues`.

In [5]:
CATEGORICAL = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
NUMERIC_ALL = [c for c in X.columns if c not in CATEGORICAL]

FEATURE_SETS = {
    "con_PageValues": NUMERIC_ALL,
    "sin_PageValues": [c for c in NUMERIC_ALL if c != "PageValues"],
}

#### Preprocesador

Las variables numéricas pasan sin cambios, porque los árboles no dependen de la escala. Las categóricas se codifican en one-hot; los códigos con menos de 30 sesiones se agrupan en una sola columna.

In [6]:
def make_preprocessor(numeric):
    return ColumnTransformer(
        [
            ("num", "passthrough", numeric),
            ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30,
                                  sparse_output=False), CATEGORICAL),
        ],
        verbose_feature_names_out=False,
    )

#### Modelos

El modelo de referencia siempre predice la clase más frecuente ("no compra").

In [7]:
MODELS = {
    "referencia": (
        [("model", DummyClassifier(strategy="most_frequent"))],
        {},
    ),
    "arbol": (
        [("model", DecisionTreeClassifier(random_state=SEED))],
        {"model__max_depth": [3, 5, 8, None], "model__min_samples_leaf": [1, 5, 20]},
    ),
    "rf": (
        [("model", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))],
        {"model__max_depth": [10, 20, None], "model__min_samples_leaf": [1, 5]},
    ),
}

#### Validación cruzada

Cinco pliegues estratificados. Para cada modelo, `GridSearchCV` elige los parámetros con mayor F1 de la clase "compra".

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = {
    "f1": make_scorer(f1_score, zero_division=0),
    "recall": "recall",
    "precision": make_scorer(precision_score, zero_division=0),
    "average_precision": "average_precision",
}

rows = []
for set_name, numeric in FEATURE_SETS.items():
    for model_name, (steps, grid) in MODELS.items():
        pipe = Pipeline([("preprocessor", make_preprocessor(numeric))] + steps)
        search = GridSearchCV(pipe, grid, cv=cv, scoring=SCORING, refit="f1", n_jobs=-1)
        search.fit(X, y)
        i = search.best_index_
        res = search.cv_results_
        rows.append({
            "modelo": model_name,
            "variables": set_name,
            "parametros": str(search.best_params_),
            **{f"{m}_mean": res[f"mean_test_{m}"][i] for m in SCORING},
            **{f"{m}_std": res[f"std_test_{m}"][i] for m in SCORING},
        })
        print(f"{set_name:15} | {model_name:20} | F1 = {res['mean_test_f1'][i]:.3f}")

con_PageValues  | referencia           | F1 = 0.000
con_PageValues  | arbol                | F1 = 0.633
con_PageValues  | rf                   | F1 = 0.622
sin_PageValues  | referencia           | F1 = 0.000
sin_PageValues  | arbol                | F1 = 0.292
sin_PageValues  | rf                   | F1 = 0.120


#### Resultados

In [9]:
cv_results = pd.DataFrame(rows).round(4)
cv_results.sort_values("f1_mean", ascending=False)

,modelo,variables,parametros,f1_mean,recall_mean,precision_mean,average_precision_mean,f1_std,recall_std,precision_std,average_precision_std
1,arbol,con_PageValues,"{'model__max_depth': 8, 'model__min_samples_le...",0.6332,0.5806,0.6971,0.6752,0.0127,0.0119,0.0269,0.0184
2,rf,con_PageValues,"{'model__max_depth': 20, 'model__min_samples_l...",0.6223,0.5262,0.7615,0.7355,0.0106,0.0094,0.0195,0.0146
4,arbol,sin_PageValues,"{'model__max_depth': None, 'model__min_samples...",0.2925,0.3099,0.2771,0.1935,0.0305,0.0378,0.0247,0.0118
5,rf,sin_PageValues,"{'model__max_depth': None, 'model__min_samples...",0.1205,0.0688,0.5008,0.3591,0.0247,0.0156,0.0388,0.0177
0,referencia,con_PageValues,{},0.0000,0.0000,0.0000,0.1547,0.0000,0.0000,0.0000,0.0002
3,referencia,sin_PageValues,{},0.0000,0.0000,0.0000,0.1547,0.0000,0.0000,0.0000,0.0002


#### Guardado de resultados

In [10]:
cv_results.to_csv(RESULTS_DIR / "cv_unbalanced.csv", index=False)
print(f"Guardado: {RESULTS_DIR / 'cv_unbalanced.csv'}")

Guardado: results\cv_unbalanced.csv
